<a href="https://colab.research.google.com/github/kurniarahmi/Sales-Revenue-Analysis/blob/main/python/Sales_%26_Revenue_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install google-cloud-bigquery pandas matplotlib seaborn

In [2]:
pip install pyarrow

In [ ]:
from google.colab import auth
auth.authenticate_user()

In [ ]:
from google.cloud import bigquery
import pandas as pd
client = bigquery.Client(project='fair-hallway-256512')

## Dataset yang digunakan

In [ ]:
query = """
SELECT
  oi.order_id,
  oi.user_id,
  oi.product_id,
  oi.sale_price,
  oi.created_at,
  u.country,
  p.name
FROM
  bigquery-public-data.thelook_ecommerce.order_items oi
JOIN
  bigquery-public-data.thelook_ecommerce.users u
ON
  oi.user_id = u.id
JOIN
  bigquery-public-data.thelook_ecommerce.products p
ON
  oi.product_id = p.id
"""

df = client.query(query).to_dataframe()

In [ ]:
df

## Cleaning Data

In [ ]:
df['created_at'] = pd.to_datetime(df['created_at'])
df = df.dropna()

In [ ]:
df['month'] = df['created_at'].dt.to_period('M')

#Exploratory Data Analysis (EDA)

## Revenue Trend

In [ ]:
monthly_revenue = df.groupby('month')['sale_price'].sum()
monthly_revenue.plot()

* Revenue menunjukkan tren naik konsisten tanpa penurunan signifikan
* Di periode akhir terlihat akselerasi growth (makin cepat naik)
* Volatilitas rendah → bisnis relatif stabil
* Belum ada tanda plateau → masih ada potensi growth lebih lanjut

##Top Product Contribution

In [ ]:
top_products = df.groupby('name')['sale_price'].sum().sort_values(ascending=False).head(10)

In [ ]:
top_products

* Revenue didominasi oleh produk premium (outerwear & brand besar) → kontribusi tinggi meskipun jumlah produk sedikit
* Top products menunjukkan pola high-value, low-volume (harga tinggi jadi driver utama revenue)
* Ada sedikit variasi (jeans, shorts), tapi tetap kalah kontribusi dibanding jacket premium

In [ ]:
worst_products = df.groupby('name')['sale_price'].sum().sort_values(ascending=True).head(10)
worst_products

* Worst products punya revenue sangat kecil → indikasi low price, low impact, bahkan berpotensi tidak profitable
* Terdapat long-tail distribution → sebagian kecil produk menyumbang mayoritas revenue

##Customer Value Analysis

In [ ]:
customer_value = df.groupby('user_id')['sale_price'].sum()
customer_value.describe()

* Distribusi nilai transaksi menunjukkan right-skewed (condong ke kanan) → sebagian besar transaksi bernilai rendah, tapi ada sedikit transaksi dengan nilai sangat tinggi (max 1936)
* Median (89) jauh lebih rendah dari mean (134) → menandakan outlier high-value cukup signifikan dan mendorong rata-rata naik
50% transaksi berada di range 41 – 180 → ini adalah core spending customer (mid-range)
* Adanya nilai minimum sangat kecil (0.02) → indikasi anomali data atau diskon ekstrem

👉 mayoritas customer belanja di level rendah–menengah, tapi revenue sangat dipengaruhi oleh transaksi high-value (premium segment)

#Deep Analysis

##RFM Analysis

In [ ]:
import datetime as dt

snapshot_date = df['created_at'].max()

rfm = df.groupby('user_id').agg({
    'created_at': lambda x: (snapshot_date - x.max()).days,
    'order_id': 'nunique',
    'sale_price': 'sum'
})

rfm.columns = ['recency', 'frequency', 'monetary']


In [ ]:
rfm

## Distribusi RFM (Recency, Frequency, Monetary)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(15, 5))

# Recency distribution
plt.subplot(1, 3, 1)
sns.histplot(rfm['recency'], bins=30, kde=True)
plt.title('Distribusi Recency')
plt.xlabel('Recency (Hari)')
plt.ylabel('Jumlah Pelanggan')

# Frequency distribution
plt.subplot(1, 3, 2)
sns.histplot(rfm['frequency'], bins=30, kde=True)
plt.title('Distribusi Frequency')
plt.xlabel('Frequency (Jumlah Pesanan)')
plt.ylabel('Jumlah Pelanggan')

# Monetary distribution
plt.subplot(1, 3, 3)
sns.histplot(rfm['monetary'], bins=30, kde=True)
plt.title('Distribusi Monetary')
plt.xlabel('Monetary (Total Pembelian)')
plt.ylabel('Jumlah Pelanggan')

plt.tight_layout()
plt.show()

* Recency: Distribusi condong ke kanan → banyak customer sudah lama tidak bertransaksi → indikasi churn cukup tinggi / engagement menurun
* Frequency: Mayoritas customer hanya melakukan 1–2 transaksi → menunjukkan repeat rate masih rendah
* Monetary: Distribusi sangat skewed → sebagian besar customer spend kecil, hanya sedikit yang high-value

Singkatnya:
Customer base didominasi oleh low engagement (jarang beli & sudah lama tidak aktif). Sementara, Revenue kemungkinan besar ditopang oleh segelintir customer yang sering dan belanja besar dan ada peluang besar di retention & increasing purchase frequency

## Cohort Analysis

### Retention per Cohort (Yearly)

In [ ]:
df['order_year'] = df['created_at'].dt.to_period('Y')
df['cohort'] = df.groupby('user_id')['order_year'].transform('min')

cohort_data = df.groupby(['cohort', 'order_year'])['user_id'].nunique().reset_index()

In [ ]:
cohort_data

In [ ]:
cohort_data['period_number'] = (cohort_data.order_year - cohort_data.cohort).apply(lambda x: x.n)

cohort_pivot = cohort_data.pivot_table(index='cohort', columns='period_number', values='user_id')

cohort_sizes = cohort_pivot.iloc[:,0]
retention_matrix = cohort_pivot.divide(cohort_sizes, axis=0)

plt.figure(figsize=(15, 8))
sns.heatmap(retention_matrix, annot=True, fmt='.0%', cmap='Blues')
plt.title('Cohort Retention Rate')
plt.xlabel('Period Number (Year)')
plt.ylabel('Cohort')
plt.show()

* Retention rate turun drastis setelah periode pertama → dari 100% ke ~15–26% di tahun berikutnya → indikasi drop-off tinggi setelah pembelian awal
* Setelah tahun ke-2/ke-3, retention makin menurun hingga <10% → sangat sedikit customer yang bertahan jangka panjang
* Cohort yang lebih baru (2023–2024) sempat menunjukkan retention awal lebih tinggi (~21–26%), tapi tetap turun cepat di periode berikutnya
* Pola ini konsisten di hampir semua cohort → menunjukkan masalah struktural di retention, bukan kasus tertentu

### Retention per Cohort (Monthly per Year)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Get unique years from the DataFrame
all_years = sorted(df['created_at'].dt.year.unique())

for year in all_years:
    print(f"\n--- Cohort Analysis for Year: {year} ---")

    # Filter data for the current year
    df_year = df[df['created_at'].dt.year == year].copy()

    if df_year.empty:
        print(f"No data for year {year}, skipping.")
        continue

    # Define order_month and cohort for the current year
    df_year['order_month'] = df_year['created_at'].dt.to_period('M')
    df_year['cohort'] = df_year.groupby('user_id')['order_month'].transform('min')

    # Aggregate cohort data
    cohort_data_yearly = df_year.groupby(['cohort', 'order_month'])['user_id'].nunique().reset_index()

    # Calculate period number
    cohort_data_yearly['period_number'] = (cohort_data_yearly.order_month - cohort_data_yearly.cohort).apply(lambda x: x.n)

    # Create pivot table
    cohort_pivot_yearly = cohort_data_yearly.pivot_table(index='cohort', columns='period_number', values='user_id')

    # Calculate retention matrix
    cohort_sizes_yearly = cohort_pivot_yearly.iloc[:,0]
    retention_matrix_yearly = cohort_pivot_yearly.divide(cohort_sizes_yearly, axis=0)

    # Plot heatmap
    plt.figure(figsize=(15, 8))
    sns.heatmap(retention_matrix_yearly, annot=True, fmt='.0%', cmap='Blues')
    plt.title(f'Cohort Retention Rate (Monthly) - Year {year}')
    plt.xlabel('Period Number (Months)')
    plt.ylabel('Cohort (First Purchase Month)')
    plt.show()


### Churn Rate per Cohort (Yearly)

In [ ]:
# Calculate churn matrix (1 - retention_matrix)
churn_matrix = 1 - retention_matrix

plt.figure(figsize=(15, 8))
sns.heatmap(churn_matrix, annot=True, fmt='.0%', cmap='Reds')
plt.title('Cohort Churn Rate')
plt.xlabel('Period Number (Year)')
plt.ylabel('Cohort')
plt.show()

### Churn Rate per Cohort (Monthly per Year)

In [ ]:
for year in all_years:
    print(f"\n--- Cohort Churn Analysis for Year: {year} ---")

    df_year = df[df['created_at'].dt.year == year].copy()

    if df_year.empty:
        print(f"No data for year {year}, skipping.")
        continue

    df_year['order_month'] = df_year['created_at'].dt.to_period('M')
    df_year['cohort'] = df_year.groupby('user_id')['order_month'].transform('min')

    cohort_data_yearly = df_year.groupby(['cohort', 'order_month'])['user_id'].nunique().reset_index()

    cohort_data_yearly['period_number'] = (cohort_data_yearly.order_month - cohort_data_yearly.cohort).apply(lambda x: x.n)

    cohort_pivot_yearly = cohort_data_yearly.pivot_table(index='cohort', columns='period_number', values='user_id')

    cohort_sizes_yearly = cohort_pivot_yearly.iloc[:,0]
    retention_matrix_yearly = cohort_pivot_yearly.divide(cohort_sizes_yearly, axis=0)

    # Calculate churn matrix for the yearly-monthly analysis
    churn_matrix_yearly = 1 - retention_matrix_yearly

    plt.figure(figsize=(15, 8))
    sns.heatmap(churn_matrix_yearly, annot=True, fmt='.0%', cmap='Reds')
    plt.title(f'Cohort Churn Rate (Monthly) - Year {year}')
    plt.xlabel('Period Number (Months)')
    plt.ylabel('Cohort (First Purchase Month)')
    plt.show()

##Churn Indication

In [ ]:
last_purchase = df.groupby('user_id')['created_at'].max()
inactive_users = last_purchase[last_purchase < (snapshot_date - pd.Timedelta(days=90))]

In [ ]:
inactive_users

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 6))
sns.histplot(inactive_users, kde=True, bins=30)
plt.title('Distribution of Last Purchase Dates for Inactive Users')
plt.xlabel('Last Purchase Date')
plt.ylabel('Number of Inactive Users')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

###Calculate the percentage of inactive users by cohort year

In [ ]:
import pandas as pd

# Get the cohort year for each user (first purchase year)
user_cohort = df.groupby('user_id')['cohort'].first().rename('cohort_year')

# Convert inactive_users Series to a DataFrame for easier merging
inactive_users_df = inactive_users.reset_index()
inactive_users_df = inactive_users_df.rename(columns={'created_at': 'last_purchase_date'})

# Merge with user_cohort to get the cohort year of inactive users
inactive_users_with_cohort = inactive_users_df.merge(user_cohort, on='user_id', how='left')

# Count total users per cohort
total_users_per_cohort = user_cohort.value_counts().sort_index()

# Count inactive users per cohort
inactive_users_per_cohort = inactive_users_with_cohort['cohort_year'].value_counts().sort_index()

# Calculate percentage of inactive users per cohort
percentage_inactive_by_cohort = (inactive_users_per_cohort / total_users_per_cohort * 100).fillna(0)

print("Percentage of Inactive Users by Cohort Year:")
print(percentage_inactive_by_cohort)

# Visualize the percentage
plt.figure(figsize=(10, 6))
sns.barplot(x=percentage_inactive_by_cohort.index.astype(str), y=percentage_inactive_by_cohort.values, palette='viridis')
plt.title('Percentage of Inactive Users by Cohort Year')
plt.xlabel('Cohort Year')
plt.ylabel('Percentage Inactive (%)')
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
rfm['is_inactive'] = rfm.index.isin(inactive_users.index)

plt.figure(figsize=(8, 6))
sns.boxplot(x='is_inactive', y='frequency', data=rfm)
plt.title('Purchase Frequency Distribution by Inactivity Status')
plt.xlabel('Is Inactive (True = Inactive, False = Active)')
plt.ylabel('Frequency (Number of Orders)')
plt.xticks([0, 1], ['Active', 'Inactive'])
plt.show()

In [ ]:
mean_frequency_by_status = rfm.groupby('is_inactive')['frequency'].mean()
print("Average Purchase Frequency by Inactivity Status:")
print(mean_frequency_by_status)

#ML Project Extension [Predicting Customer Churn Using Machine Learning]
Membangun model machine learning untuk memprediksi kemungkinan customer akan berhenti bertransaksi (churn), sehingga bisnis dapat melakukan tindakan preventif.

##Definisi Churn

In [ ]:
df['last_purchase'] = df.groupby('user_id')['created_at'].transform('max')

snapshot_date = df['created_at'].max()

df['churn'] = (snapshot_date - df['last_purchase']).dt.days > 90

##Feature Engineering

In [ ]:
rfm = df.groupby('user_id').agg({
    'created_at': lambda x: (snapshot_date - x.max()).days,
    'order_id': 'nunique',
    'sale_price': 'sum'
})

rfm.columns = ['recency', 'frequency', 'monetary']

In [ ]:
rfm['avg_order_value'] = rfm['monetary'] / rfm['frequency']

##Prepare Data

In [ ]:
import datetime as dt
from sklearn.model_selection import train_test_split

# Ensure snapshot_date is available
snapshot_date = df['created_at'].max()

# 1. Recalculate RFM from a clean state to prevent merge errors
rfm_clean = df.groupby('user_id').agg({
    'created_at': lambda x: (snapshot_date - x.max()).days,
    'order_id': 'nunique',
    'sale_price': 'sum'
})
rfm_clean.columns = ['recency', 'frequency', 'monetary']

# 2. Calculate avg_order_value
rfm_clean['avg_order_value'] = rfm_clean['monetary'] / rfm_clean['frequency']

# Get the churn status for each user from the original df
# This 'churn' column is derived from `df['churn']` created earlier.
user_churn_status = df.groupby('user_id')['churn'].first()

# 3. Add the churn status to the rfm DataFrame
# Since 'rfm_clean' does not have a 'churn' column, this merge will add a single 'churn' column without suffixes.
rfm = rfm_clean.merge(user_churn_status, on='user_id', how='left')

# 'recency' is explicitly removed from features to prevent data leakage in churn prediction.
# The 'churn' column is the target variable and MUST be removed from the feature set 'X'.
X = rfm.drop(columns=['recency', 'churn'], errors='raise') # Drop recency and the 'churn' column from features
y = rfm['churn'] # 'churn' is the correct target variable

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# Retrain the model with the updated X_train and y_train
model = RandomForestClassifier(random_state=42) # Added random_state for reproducibility
model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = model.predict(X_test)

print("Model Evaluation (after addressing data leakage and removing 'recency'):")
print(classification_report(y_test, y_pred))

# Display feature importances again
importance = pd.Series(model.feature_importances_, index=X.columns)
print("\nFeature Importances (after addressing data leakage):")
print(importance.sort_values(ascending=False))

##Model Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

##Evaluation

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

In [ ]:
importance = pd.Series(model.feature_importances_, index=X.columns)
importance.sort_values(ascending=False)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.histplot(data=rfm, x='recency', hue='churn', kde=True, bins=50)
plt.title('Distribution of Recency for Churned vs. Active Users')
plt.xlabel('Recency (Days)')
plt.ylabel('Number of Users')
plt.legend(title='Churn Status', labels=['Active', 'Churned'])
plt.show()